In [1109]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import talib.abstract as ta
import scipy as sp
from sklearn.preprocessing import PowerTransformer
from typing import List
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [1110]:
# Apple Futures data using yfinance
ticker = 'TUPRS.IS'
data = yf.download(ticker, start="2015-01-01", interval="1D")

# Check the data
display(data.head())

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,TUPRS.IS,TUPRS.IS,TUPRS.IS,TUPRS.IS,TUPRS.IS
Date,,,,,
2015-01-01,3.433072,3.433072,3.433072,3.433072,0
2015-01-02,3.470320,3.473425,3.426864,3.436176,2724008
2015-01-05,3.513777,3.544817,3.513777,3.516881,4698064
2015-01-06,3.519985,3.523089,3.476528,3.479632,3890747
2015-01-07,3.501361,3.526193,3.488945,3.519986,5780691


In [1111]:
data.columns = ['Close','High','Low','Open','Volume']

In [1112]:
data_copy = data.copy()

# Preprocess
## Previous High & Low columns

In [1113]:
data_copy['previous_high'] = data_copy['High'].shift(1)
data_copy['previous_low'] = data_copy['Low'].shift(1) 
data_copy['previous_close'] = data_copy['Close'].shift(1)

## Gerekli Göstergeler

### ATR
14 günlük ATR 

In [1114]:
data_copy['ATR'] = ta.ATR(data_copy['High'], data_copy['Low'], data_copy['Close'], timeperiod=14)

### RSI 
14 günlük RSI değerleri

In [1115]:
data_copy['RSI'] = ta.RSI(data_copy['Close'], timeperiod = 14)

### MACD

Short period : 12

Long period: 26

Signal period : 9

In [1116]:
data_copy['MACD'], data_copy['MACDSignal'],data_copy['MACDHist'] = ta.MACD(data_copy['Close'],
                                                                           fastperiod=12,
                                                                           slowperiod=26,
                                                                           signalperiod=9
)

### SMA 
20-50 periyotluk SMA

In [1117]:
data_copy['MA20'] = ta.SMA(data_copy['Close'], timeperiod = 20)
data_copy['MA50'] = ta.SMA(data_copy['Close'], timeperiod = 50)

### Volume Filter
Eğer volumne değerleri, volume'un 20 günlük hareketli ortalamasından yüzde 20 fazla ise yüksek volume ve yüzde 80 fazla ise çok yüksek volume olarak etiketlenir.

In [1118]:
data_copy["VOL_MA20"] = ta.SMA(data_copy['Volume'], 20)
data_copy["high_volume"] = data_copy["Volume"] > 1.2 * data_copy["VOL_MA20"]
data_copy["very_high_volume"] = data_copy["Volume"] > 1.8 * data_copy["VOL_MA20"]

### Bollinger band 

In [1119]:
data_copy['UpperBB'], data_copy['MiddleBB'], data_copy['LowerBB'] = ta.BBANDS(
    data_copy['Close'], 
    timeperiod=20,     
    nbdevup=2.0,       
    nbdevdn=2.0,      
    matype=0           
)

## Gap Detection

In [1120]:
data_copy['gap_up'] = (data_copy['Low'] > data_copy['previous_high'])
data_copy['gap_down'] = (data_copy['High'] < data_copy['previous_low'])
data_copy["is_gap"] = data_copy["gap_up"] | data_copy["gap_down"]

## Raw Gap Size

In [1121]:
data_copy["gap_size"] = 0.0
data_copy.loc[data_copy["gap_up"], "gap_size"] = data_copy["Low"] - data_copy["previous_high"]
data_copy.loc[data_copy["gap_down"], "gap_size"] = data_copy["previous_low"] - data_copy["High"]

## Gap Normalization
https://cssanalytics.wordpress.com/2011/03/21/normalized-gaplap-indicator-ngiungid-nliu-nlid/

gapi Atr ile normalize ediyorum. Aslında bu yazının uygulamalar kısmından da anlaşılacağı için bu common gap ve diğerlerini ayırt edecek bir gösterge.

In [1122]:
data_copy["normalized_gap_size"] = data_copy["gap_size"] / data_copy["ATR"]

## Gap Direction
Gap up ve gap down sütublarını tek sütuna indirme 

In [1123]:
data_copy["gap_direction"] = 0
data_copy.loc[data_copy["gap_up"], "gap_direction"] = 1
data_copy.loc[data_copy["gap_down"], "gap_direction"] = -1

## Trend Anaylsis

Burada kullandığım strateji double ma stratejisi. Eğer kapanış fiyatı iki ma üzerinde ise up trend altında ise down trend olur. 

In [1124]:
data_copy["trend_short"] = 0
data_copy.loc[(data_copy["Close"] > data_copy["MA20"]) & (data_copy["MA20"] > data_copy["MA50"]), "trend_short"] = 1
data_copy.loc[(data_copy["Close"] < data_copy["MA20"]) & (data_copy["MA20"] < data_copy["MA50"]), "trend_short"] = -1

In [1125]:
# ==========================================
# GÖRSELLEŞTİRME (PLOTLY)
# ==========================================

print("\nGörselleştirme hazırlanıyor...")

# 1. Grafik Verisini Hazırla (Son dönem daha okunaklı olur)
plot_data_copy = data_copy.copy()  # Son 250 günü (yaklaşık 1 yıl) çizelim
# Eğer veri çoksa grafiği kasmaması için son kısmı seçmek iyidir.

# 2. Subplot oluştur (2 satır, 1 sütun: Üstte mumlar, altta hacim)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                    vertical_spacing=0.03, subplot_titles=(f'{ticker} Technical Analysis', 'Volume'), 
                    row_width=[0.2, 0.7]) # Alt grafiği küçült


# --- A. ÜST GRAFİK (CANDLESTICK & GÖSTERGELER) ---

# Candlestick (Mum Grafiği)
fig.add_trace(go.Candlestick(x=plot_data_copy.index,
                open=plot_data_copy['Open'],
                high=plot_data_copy['High'],
                low=plot_data_copy['Low'],
                close=plot_data_copy['Close'],
                name='Price'), row=1, col=1)

# Hareketli Ortalamalar
fig.add_trace(go.Scatter(x=plot_data_copy.index, y=plot_data_copy['MA20'], name='MA20', line=dict(color='blue', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=plot_data_copy.index, y=plot_data_copy['MA50'], name='MA50', line=dict(color='orange', width=1)), row=1, col=1)


# --- B. ARKA PLAN TREND RENKLENDİRMESİ ---

# Arka planı trend durumuna göre renklendirme (vrect)
current_trend = None
start_date = plot_data_copy.index[0]

for i in range(len(plot_data_copy)):
    date = plot_data_copy.index[i]
    trend = plot_data_copy['trend_short'].iloc[i]
    
    # Trend değiştiyse veya son barsak, bir önceki bölgeyi çiz
    if trend != current_trend or i == len(plot_data_copy) - 1:
        color = None
        if current_trend == 1: # Uptrend
            color = "rgba(0, 250, 0, 0.1)" # Çok açık yeşil
        elif current_trend == -1: # Downtrend
            color = "rgba(250, 0, 0, 0.1)" # Çok açık kırmızı
        
        # Renk varsa vrect ekle
        if color:
            fig.add_vrect(x0=start_date, x1=date, fillcolor=color, layer="below", line_width=0, row=1, col=1)
        
        # Yeni bölgeyi başlat
        start_date = date
        current_trend = trend




# --- D. ALT GRAFİK (HACİM) ---

# Hacim barları
fig.add_trace(go.Bar(x=plot_data_copy.index, y=plot_data_copy['Volume'], name='Volume', marker_color='gray'), row=2, col=1)
# Hacim MA
fig.add_trace(go.Scatter(x=plot_data_copy.index, y=plot_data_copy['VOL_MA20'], name='Volume MA20', line=dict(color='black', width=1)), row=2, col=1)


# --- E. GRAFİK AYARLARI ---

fig.update_layout(
    title=f'{ticker} Stocks Analysis (Gaps & Trends)',
    xaxis_title='Date',
    yaxis_title='Price',
    xaxis_rangeslider_visible=False, # Alttaki slider'ı kapat (subplot'ta çakışıyor)
    height=800, # Grafik yüksekliği
    template='plotly_white' # Temiz beyaz tema
)

# Y eksenlerini ayarla (üst grafik fiyat, alt grafik hacim)
fig.update_yaxes(title_text="Price ($)", row=1, col=1)
fig.update_yaxes(title_text="Volume", row=2, col=1)

# Grafiği göster
fig.show()

# İsteğe bağlı: Grafiği HTML dosyası olarak kaydet
# fig.write_html(f"{ticker}_gap_analizi.html")
# print(f"Grafik {ticker}_gap_analizi.html olarak kaydedildi.")


Görselleştirme hazırlanıyor...


## Momentum

https://tr.tradingview.com/support/solutions/43000502338/

RSI değeri 30'un altında ise aşırı satım ve 70'in üzerinde ise aşırı satım olur ama bu trend reversal için diğer bi common strategy olan trendin devam edip edemeyeceği 45-55 stratejisi ile hesaplanır. 

https://www.investopedia.com/terms/m/macd.asp

If MACD is above the signal line, the histogram will be above the MACD’s baseline or zero line. If MACD is below its signal line, the histogram will be below the MACD’s baseline. Traders use the MACD’s histogram to identify peaks of bullish or bearish momentum, and to generate overbought/oversold trade signals.

In [1126]:
data_copy["bullish_momentum"] = (data_copy["RSI"] > 55) & (data_copy["MACD"] > data_copy["MACDSignal"])
data_copy["bearish_momentum"] = (data_copy["RSI"] < 45) & (data_copy["MACD"] < data_copy["MACDSignal"])

In [1127]:
plot_data_copy = data_copy.copy().iloc[-500:,:] 

# 1. Subplot oluştur (3 satır: Üstte mumlar, ortada hacim, altta RSI)
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, 
                    vertical_spacing=0.03, 
                    subplot_titles=(f'{ticker} Technical Analysis', 'Volume', 'RSI (14)'), 
                    row_width=[0.2, 0.2, 0.6]) # RSI ve Hacim %20, Ana grafik %60

# --- A. ÜST GRAFİK (CANDLESTICK & GÖSTERGELER) ---
# (Mevcut kodun aynısı burada kalacak)
fig.add_trace(go.Candlestick(x=plot_data_copy.index,
                open=plot_data_copy['Open'],
                high=plot_data_copy['High'],
                low=plot_data_copy['Low'],
                close=plot_data_copy['Close'],
                name='Price'), row=1, col=1)

fig.add_trace(go.Scatter(x=plot_data_copy.index, y=plot_data_copy['MA20'], name='MA20', line=dict(color='blue', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=plot_data_copy.index, y=plot_data_copy['MA50'], name='MA50', line=dict(color='orange', width=1)), row=1, col=1)


# Arka planı trend durumuna göre renklendirme (vrect)
current_trend = None
start_date = plot_data_copy.index[0]

for i in range(len(plot_data_copy)):
    date = plot_data_copy.index[i]
    trend = plot_data_copy['trend_short'].iloc[i]
    
    # Trend değiştiyse veya son barsak, bir önceki bölgeyi çiz
    if trend != current_trend or i == len(plot_data_copy) - 1:
        color = None
        if current_trend == 1: # Uptrend
            color = "rgba(0, 250, 0, 0.1)" # Çok açık yeşil
        elif current_trend == -1: # Downtrend
            color = "rgba(250, 0, 0, 0.1)" # Çok açık kırmızı
        
        # Renk varsa vrect ekle
        if color:
            fig.add_vrect(x0=start_date, x1=date, fillcolor=color, layer="below", line_width=0, row=1, col=1)
        
        # Yeni bölgeyi başlat
        start_date = date
        current_trend = trend

# --- B. ARKA PLAN TREND RENKLENDİRMESİ ---
# (Mevcut trend döngün burada devam edebilir)

# --- D. ORTA GRAFİK (HACİM) ---
fig.add_trace(go.Bar(x=plot_data_copy.index, y=plot_data_copy['Volume'], name='Volume', marker_color='gray'), row=2, col=1)
fig.add_trace(go.Scatter(x=plot_data_copy.index, y=plot_data_copy['VOL_MA20'], name='Volume MA20', line=dict(color='black', width=1)), row=2, col=1)

# --- E. ALT GRAFİK (RSI) ---
# RSI Çizgisi
fig.add_trace(go.Scatter(x=plot_data_copy.index, y=plot_data_copy['RSI'], 
                         name='RSI', line=dict(color='purple', width=1.5)), row=3, col=1)

# RSI Eşik Değerleri (70 ve 30)
fig.add_hline(y=65, line_dash="dash", line_color="red", line_width=1, row=3, col=1)
fig.add_hline(y=45, line_dash="dash", line_color="green", line_width=1, row=3, col=1)

# RSI Arka Plan Renklendirme (Opsiyonel: 30-70 arasını hafif gri yapar)
fig.add_hrect(y0=45, y1=65, fillcolor="gray", opacity=0.1, layer="below", line_width=0, row=3, col=1)

# --- E. ALT GRAFİK (RSI) ---


# RSI Eşik Değerleri (70 ve 30)
fig.add_hline(y=70, line_dash="dash", line_color="darkred", line_width=1, row=3, col=1)
fig.add_hline(y=30, line_dash="dash", line_color="darkgreen", line_width=1, row=3, col=1)



# --- F. GRAFİK AYARLARI ---
fig.update_layout(
    title=f'{ticker} Stocks Analysis (Price, Volume & RSI)',
    xaxis_rangeslider_visible=False,
    height=900, # 3 grafik olduğu için yüksekliği biraz artırdık
    template='plotly_white',
    showlegend=True
)

# Y ekseni başlıkları
fig.update_yaxes(title_text="Price ($)", row=1, col=1)
fig.update_yaxes(title_text="Volume", row=2, col=1)
fig.update_yaxes(title_text="RSI", range=[0, 100], row=3, col=1)

fig.show()

In [1128]:

plot_data_copy = data_copy.copy().iloc[-500:,:] 

# 1. Subplot oluştur
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, 
                    vertical_spacing=0.03, 
                    subplot_titles=(f'{ticker} Technical Analysis', 'Volume', 'RSI (14)'), 
                    row_width=[0.2, 0.2, 0.6])

# --- A. ÜST GRAFİK (CANDLESTICK) ---
fig.add_trace(go.Candlestick(x=plot_data_copy.index,
                open=plot_data_copy['Open'],
                high=plot_data_copy['High'],
                low=plot_data_copy['Low'],
                close=plot_data_copy['Close'],
                name='Price'), row=1, col=1)

fig.add_trace(go.Scatter(x=plot_data_copy.index, y=plot_data_copy['MA20'], name='MA20', line=dict(color='blue', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=plot_data_copy.index, y=plot_data_copy['MA50'], name='MA50', line=dict(color='orange', width=1)), row=1, col=1)

# --- B. MOMENTUM TABANLI ARKA PLAN RENKLENDİRMESİ ---
# Momentuma göre bölgeleri belirle
start_date = plot_data_copy.index[0]
# Mevcut momentum durumunu takip etmek için (Bull, Bear veya None)
current_state = None 

for i in range(len(plot_data_copy)):
    date = plot_data_copy.index[i]
    is_bull = plot_data_copy['bullish_momentum'].iloc[i]
    is_bear = plot_data_copy['bearish_momentum'].iloc[i]
    
    # Mevcut satırdaki durumu belirle
    new_state = "bull" if is_bull else ("bear" if is_bear else None)
    
    # Durum değiştiyse veya son satırdaysak önceki bölgeyi çiz
    if new_state != current_state or i == len(plot_data_copy) - 1:
        color = None
        if current_state == "bull":
            color = "rgba(0, 255, 0, 0.15)" # Güçlü Boğa (Yeşil)
        elif current_state == "bear":
            color = "rgba(255, 0, 0, 0.15)" # Güçlü Ayı (Kırmızı)
        

        if color:
            fig.add_vrect(x0=start_date, x1=date, fillcolor=color, 
                          layer="below", line_width=0, row=1, col=1)
        
        start_date = date
        current_state = new_state

# --- C. ORTA GRAFİK (HACİM) ---
fig.add_trace(go.Bar(x=plot_data_copy.index, y=plot_data_copy['Volume'], name='Volume', marker_color='gray'), row=2, col=1)
fig.add_trace(go.Scatter(x=plot_data_copy.index, y=plot_data_copy['VOL_MA20'], name='Volume MA20', line=dict(color='black', width=1)), row=2, col=1)

# --- D. ALT GRAFİK (RSI) ---
fig.add_trace(go.Scatter(x=plot_data_copy.index, y=plot_data_copy['RSI'], 
                         name='RSI', line=dict(color='purple', width=1.5)), row=3, col=1)

# RSI Eşik Çizgileri
fig.add_hline(y=70, line_dash="dash", line_color="darkred", line_width=1, row=3, col=1)
fig.add_hline(y=30, line_dash="dash", line_color="darkgreen", line_width=1, row=3, col=1)
fig.add_hrect(y0=30, y1=70, fillcolor="gray", opacity=0.05, layer="below", line_width=0, row=3, col=1)

# --- E. GRAFİK AYARLARI ---
fig.update_layout(
    title=f'{ticker} Momentum & Trend Analysis',
    xaxis_rangeslider_visible=False,
    height=900,
    template='plotly_white',
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.update_yaxes(title_text="Price ($)", row=1, col=1)
fig.update_yaxes(title_text="Volume", row=2, col=1)
fig.update_yaxes(title_text="RSI", range=[0, 100], row=3, col=1)

fig.show()

# Deneme 

In [1129]:
def chop_index(df, n=14):
    df = df.copy()

    # --- True Range ---
    df['TR'] = ta.TRANGE(df['High'], df['Low'], df['Close'])

    # --- SUM(TR, n) ---
    df['SUM_TR'] = df['TR'].rolling(n).sum()

    # --- MaxHi(n) & MinLo(n) ---
    df['HH'] = df['High'].rolling(n).max()
    df['LL'] = df['Low'].rolling(n).min()

    # --- Denominator ---
    denom = df['HH'] - df['LL']
    denom = denom.replace(0, np.nan)

    # --- CHOP ---
    df['CHOP'] = 100 * np.log10(df['SUM_TR'] / denom) / np.log10(n)

    return df['CHOP']

In [1130]:
def classify_trend(df):
    df = df.copy()

    # Indicators
    #df['OHLC4'] = (df['High'] + df['Low'] + 2*df['Close'])/4
    df['EMA20'] = ta.EMA(df['Close'], 20)
    df['EMA50'] = ta.EMA(df['Close'], 50)
    df['EMA200'] = ta.EMA(df['Close'], 120)
    df['ADX'] = ta.ADX(df['High'], df['Low'], df['Close'], timeperiod=14)

    # Slope (EMA50 farkı)
    df['EMA50_slope'] = df['EMA50'].diff()

    df['CHOP'] = chop_index(df, 14)

    trend = []

    for i in range(len(df)):
        row = df.iloc[i]

        price = row['Close']
        ema20 = row['EMA20']
        ema50 = row['EMA50']
        slope = row['EMA50_slope']
        adx_val = row['ADX']

        # Default
        label = "SIDEWAYS"

        if adx_val >= 25 :
            # UPTREND ZONE
            if price > ema50 and slope > 0:

                # RETRACEMENT (UP)
                if price < ema20:
                    label = "RETRACEMENT_UP"
                else:
                    label = "UPTREND"

            # DOWNTREND ZONE
            elif price < ema50 and slope < 0:

                # RETRACEMENT (DOWN)
                if price > ema20:
                    label = "RETRACEMENT_DOWN"
                else:
                    label = "DOWNTREND"
        else:
            label = "SIDEWAYS"

        trend.append(label)
        

    df['Trend'] = trend
    trend_map = {
        "UPTREND": 1,
        "RETRACEMENT_UP": 0.5,
        "DOWNTREND": -1,
        "RETRACEMENT_DOWN": -0.5,
        "SIDEWAYS": 0
    }

    df['Trend_num'] = df['Trend'].map(trend_map)
    return df

In [1131]:
data_copy_copy = classify_trend(data_copy)


In [1132]:
data_copy_copy.tail()

,Close,High,Low,Open,Volume,previous_high,previous_low,previous_close,ATR,RSI,MACD,MACDSignal,MACDHist,MA20,MA50,VOL_MA20,high_volume,very_high_volume,UpperBB,MiddleBB,LowerBB,gap_up,gap_down,is_gap,gap_size,normalized_gap_size,gap_direction,trend_short,bullish_momentum,bearish_momentum,EMA20,EMA50,EMA200,ADX,EMA50_slope,CHOP,Trend,Trend_num
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-03-31,258.25,262.75,241.50,245.199997,52767272,251.25,244.50,247.00,11.314411,63.276187,7.703271,8.684654,-0.981382,246.104507,228.354356,47659339.25,False,False,266.437721,246.104507,225.771294,False,False,False,0.0,0.0,0,1,False,False,243.860686,229.318045,207.355809,31.606892,1.180896,61.531575,UPTREND,1.0
2026-04-01,253.75,257.75,251.50,253.250000,44303532,262.75,241.50,258.25,10.988381,59.391312,7.573510,8.462425,-0.888915,247.504870,229.070120,47034588.95,False,False,265.790548,247.504870,229.219192,False,False,False,0.0,0.0,0,1,False,False,244.802525,230.276161,208.122656,31.581466,0.958116,60.738362,UPTREND,1.0
2026-04-02,256.00,261.50,253.00,260.000000,36737138,257.75,251.50,253.75,10.810640,60.690839,7.565025,8.282945,-0.717920,249.209853,229.942314,46883440.85,False,False,263.564403,249.209853,234.855304,False,False,False,0.0,0.0,0,1,False,False,245.868951,231.284939,208.914017,31.809484,1.008778,58.942131,UPTREND,1.0
2026-04-03,255.25,261.25,255.25,257.000000,19513602,261.50,253.00,256.00,10.467023,60.001564,7.412338,8.108824,-0.696486,250.118457,230.691919,45574665.20,False,False,263.557609,250.118457,236.679306,False,False,False,0.0,0.0,0,1,False,False,246.762385,232.224745,209.679901,32.021216,0.939806,57.773888,UPTREND,1.0
2026-04-06,256.50,260.25,253.75,257.250000,28507877,261.25,255.25,255.25,10.183664,60.800628,7.307954,7.948650,-0.640696,250.551620,231.483816,43950365.20,False,False,264.225092,250.551620,236.878149,False,False,False,0.0,0.0,0,1,False,False,247.689776,233.176716,210.453787,31.994300,0.951971,57.118748,UPTREND,1.0


In [1133]:
plot_data_copy = data_copy_copy.copy().iloc[-500:, :] 

# =========================
# SUBPLOTS (5 rows now)
# =========================
fig = make_subplots(
    rows=5, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=(
        f'{ticker} Technical Analysis',
        'Volume',
        'RSI (14)',
        'CHOP',
        'ADX (14)'
    ),
    row_width=[0.12, 0.12, 0.18, 0.18, 0.4]
)

# =========================
# PRICE (CANDLE + EMA)
# =========================
fig.add_trace(
    go.Candlestick(
        x=plot_data_copy.index,
        open=plot_data_copy['Open'],
        high=plot_data_copy['High'],
        low=plot_data_copy['Low'],
        close=plot_data_copy['Close'],
        name='Price'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=plot_data_copy.index,
        y=plot_data_copy['EMA20'],
        name='EMA20',
        line=dict(color='blue', width=1)
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=plot_data_copy.index,
        y=plot_data_copy['EMA50'],
        name='EMA50',
        line=dict(color='orange', width=1)
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=plot_data_copy.index,
        y=plot_data_copy['EMA200'],
        name='EMA200',
        line=dict(color='purple', width=1)
    ),
    row=1, col=1
)
# =========================
# TREND BACKGROUND (vrect)
# =========================
current_trend = plot_data_copy['Trend'].iloc[0]
start_date = plot_data_copy.index[0]

for i in range(1, len(plot_data_copy)):
    date = plot_data_copy.index[i]
    trend = plot_data_copy['Trend'].iloc[i]

    if trend != current_trend or i == len(plot_data_copy) - 1:

        color = None

        if current_trend in ["UPTREND", "RETRACEMENT_UP"]:
            color = "rgba(0, 200, 0, 0.15)"

        elif current_trend in ["DOWNTREND", "RETRACEMENT_DOWN"]:
            color = "rgba(200, 0, 0, 0.15)"

        elif current_trend == "SIDEWAYS":
            color = "rgba(128, 128, 128, 0.05)"

        if color:
            fig.add_vrect(
                x0=start_date,
                x1=date,
                fillcolor=color,
                layer="below",
                line_width=0,
                row=1,
                col=1
            )

        start_date = date
        current_trend = trend

# =========================
# VOLUME
# =========================
fig.add_trace(
    go.Bar(
        x=plot_data_copy.index,
        y=plot_data_copy['Volume'],
        name='Volume',
        marker_color='gray'
    ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(
        x=plot_data_copy.index,
        y=plot_data_copy['VOL_MA20'],
        name='Volume MA20',
        line=dict(color='black', width=1)
    ),
    row=2, col=1
)

# =========================
# RSI
# =========================
fig.add_trace(
    go.Scatter(
        x=plot_data_copy.index,
        y=plot_data_copy['RSI'],
        name='RSI',
        line=dict(color='purple', width=1.5)
    ),
    row=3, col=1
)

fig.add_hline(y=70, line_dash="dash", line_color="darkred", row=3, col=1)
fig.add_hline(y=30, line_dash="dash", line_color="darkgreen", row=3, col=1)
fig.add_hrect(y0=30, y1=70, fillcolor="gray", opacity=0.1, layer="below", row=3, col=1)

# =========================
# CHOP
# =========================
fig.add_trace(
    go.Scatter(
        x=plot_data_copy.index,
        y=plot_data_copy['CHOP'],
        name='CHOP',
        line=dict(color='black', width=1.5)
    ),
    row=4, col=1
)

fig.add_hline(y=60, line_dash="dash", line_color="red", row=4, col=1)
fig.add_hline(y=40, line_dash="dash", line_color="green", row=4, col=1)

fig.add_hrect(y0=60, y1=100, fillcolor="red", opacity=0.05, layer="below", row=4, col=1)
fig.add_hrect(y0=0, y1=40, fillcolor="green", opacity=0.05, layer="below", row=4, col=1)

# =========================
# ADX
# =========================
fig.add_trace(
    go.Scatter(
        x=plot_data_copy.index,
        y=plot_data_copy['ADX'],
        name='ADX',
        line=dict(color='blue', width=1.5)
    ),
    row=5, col=1
)

fig.add_hline(y=25, line_dash="dash", line_color="red", row=5, col=1)
fig.add_hrect(y0=25, y1=100, fillcolor="green", opacity=0.05, layer="below", row=5, col=1)

# =========================
# LAYOUT
# =========================
fig.update_layout(
    title=f'{ticker} Stocks Analysis (Price, Volume, RSI, CHOP & ADX)',
    xaxis_rangeslider_visible=False,
    height=1200,
    template='plotly_white',
    showlegend=True
)

fig.update_yaxes(title_text="Price ($)", row=1, col=1)
fig.update_yaxes(title_text="Volume", row=2, col=1)
fig.update_yaxes(title_text="RSI", range=[0, 100], row=3, col=1)
fig.update_yaxes(title_text="CHOP", range=[0, 100], row=4, col=1)
fig.update_yaxes(title_text="ADX", range=[0, 100], row=5, col=1)

fig.show()

## Support & Resistance 

Pivot (Trend Reversal points) mantığına göre bi önceki kapanış fiyatının 20 günlük min ve max fiyatları ile   support ve resistance **bölgelerini** tespit etme 

In [1134]:
lookback_sr = 20

data_copy["rolling_resistance"] = data_copy["High"].rolling(lookback_sr).max().shift(1)
data_copy["rolling_support"] = data_copy["Low"].rolling(lookback_sr).min().shift(1)

# Fiyata yakınlık
data_copy["near_resistance"] = (
    (data_copy["rolling_resistance"] - data_copy["Close"]).abs() / data_copy["Close"] < 0.015 
)
data_copy["near_support"] = (
    (data_copy["Close"] - data_copy["rolling_support"]).abs() / data_copy["Close"] < 0.015
)

# Gerçek kırılım kontrolü
data_copy["breaks_resistance"] = data_copy["Close"] > data_copy["rolling_resistance"]
data_copy["breaks_support"] = data_copy["Close"] < data_copy["rolling_support"]

## Consolidation (Sideway trend)

Breakaway gap için önemli

In [1135]:
data_copy["consolidation"] = ta.ADX(data_copy['High'], data_copy['Low'], data_copy['Close'], timeperiod=14) < 25

## Trend Extension

Exhaustion gap için önemli


https://www.investopedia.com/terms/b/bollingerbands.asp

Another way to use the tool is to figure out when an asset is overbought and oversold. As the price touches or moves outside the upper band, it could be overbought, suggesting a potential selling or short opportunity. Similarly, if the price touches or falls outside the lower band, the asset may be oversold, indicating a possible buying opportunity.

In [1136]:
data_copy["extended_up_move"] = (data_copy["Close"] > data_copy["UpperBB"] )
data_copy["extended_down_move"] = (data_copy["Close"] < data_copy["LowerBB"] )

## Multi-timeframe analysis

https://www.investopedia.com/articles/trading/07/timeframes.asp

In [1137]:
# Apple Futures data using yfinance
weekly = yf.download(ticker, start="2015-01-01", interval="1wk")
weekly.columns = ['Close','High','Low','Open','Volume']

[*********************100%***********************]  1 of 1 completed


In [1138]:
weekly_copy = weekly.dropna().copy()
display(weekly_copy)

,Close,High,Low,Open,Volume
Date,,,,,
2014-12-29,3.470320,3.473424,3.426864,3.433072,2724008
2015-01-05,3.433072,3.544817,3.429968,3.516881,25057550
2015-01-12,3.445488,3.445488,3.302702,3.426863,29120448
2015-01-19,3.402031,3.551025,3.395823,3.504465,28371343
2015-01-26,3.299598,3.461008,3.293390,3.367887,45566444
...,...,...,...,...,...
2026-03-02,247.836731,253.360224,209.892735,209.892735,253779056
2026-03-09,253.120071,266.328424,230.065492,266.088272,263237190
2026-03-16,243.273834,253.120060,238.710949,249.517783,126199414


### Weekly MA 10-20 period

In [1139]:
weekly_copy['MA10'] = ta.SMA(weekly_copy['Close'], timeperiod = 10)
weekly_copy['MA20'] = ta.SMA(weekly_copy['Close'], timeperiod = 20)

### Trend Detection

In [1140]:
weekly_copy["weekly_trend_raw"] = 0
weekly_copy.loc[(weekly_copy["Close"] > weekly_copy["MA10"]) & (weekly_copy["MA10"] > weekly_copy["MA20"]), "weekly_trend_raw"] = 1
weekly_copy.loc[(weekly_copy["Close"] < weekly_copy["MA10"]) & (weekly_copy["MA10"] < weekly_copy["MA20"]), "weekly_trend_raw"] = -1

In [1141]:
weekly_copy["weekly_trend"] = weekly_copy["weekly_trend_raw"].shift(1)

In [1142]:
weekly_copy.tail()

,Close,High,Low,Open,Volume,MA10,MA20,weekly_trend_raw,weekly_trend
Date,,,,,,,,,
2026-03-02,247.836731,253.360224,209.892735,209.892735,253779056,213.264467,200.180993,1,1.0
2026-03-09,253.120071,266.328424,230.065492,266.088272,263237190,220.613115,203.624770,1,1.0
2026-03-16,243.273834,253.120060,238.710949,249.517783,126199414,225.987712,206.297660,1,1.0
2026-03-23,240.500000,265.750000,234.000000,255.250000,272082927,228.664195,208.812646,1,1.0
2026-03-30,255.250000,262.750000,241.500000,245.000000,188979896,232.498679,212.007496,1,1.0


### Haftalık trendin günlük tabloya eklenmesi

In [1143]:
weekly_trend_daily = weekly_copy["weekly_trend"].reindex(data_copy.index, method="ffill")
data_copy["weekly_trend"] = weekly_trend_daily

In [1144]:
data_copy.tail()

,Close,High,Low,Open,Volume,previous_high,previous_low,previous_close,ATR,RSI,MACD,MACDSignal,MACDHist,MA20,MA50,VOL_MA20,high_volume,very_high_volume,UpperBB,MiddleBB,LowerBB,gap_up,gap_down,is_gap,gap_size,normalized_gap_size,gap_direction,trend_short,bullish_momentum,bearish_momentum,rolling_resistance,rolling_support,near_resistance,near_support,breaks_resistance,breaks_support,consolidation,extended_up_move,extended_down_move,weekly_trend
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-03-31,258.25,262.75,241.50,245.199997,52767272,251.25,244.50,247.00,11.314411,63.276187,7.703271,8.684654,-0.981382,246.104507,228.354356,47659339.25,False,False,266.437721,246.104507,225.771294,False,False,False,0.0,0.0,0,1,False,False,266.328415,209.892735,False,False,False,False,False,False,False,1.0
2026-04-01,253.75,257.75,251.50,253.250000,44303532,262.75,241.50,258.25,10.988381,59.391312,7.573510,8.462425,-0.888915,247.504870,229.070120,47034588.95,False,False,265.790548,247.504870,229.219192,False,False,False,0.0,0.0,0,1,False,False,266.328415,219.306692,False,False,False,False,False,False,False,1.0
2026-04-02,256.00,261.50,253.00,260.000000,36737138,257.75,251.50,253.75,10.810640,60.690839,7.565025,8.282945,-0.717920,249.209853,229.942314,46883440.85,False,False,263.564403,249.209853,234.855304,False,False,False,0.0,0.0,0,1,False,False,266.328415,219.306692,False,False,False,False,False,False,False,1.0
2026-04-03,255.25,261.25,255.25,257.000000,19513602,261.50,253.00,256.00,10.467023,60.001564,7.412338,8.108824,-0.696486,250.118457,230.691919,45574665.20,False,False,263.557609,250.118457,236.679306,False,False,False,0.0,0.0,0,1,False,False,266.328415,224.878215,False,False,False,False,False,False,False,1.0
2026-04-06,256.50,260.25,253.75,257.250000,28507877,261.25,255.25,255.25,10.183664,60.800628,7.307954,7.948650,-0.640696,250.551620,231.483816,43950365.20,False,False,264.225092,250.551620,236.878149,False,False,False,0.0,0.0,0,1,False,False,266.328415,230.065488,False,False,False,False,False,False,False,1.0


## Günlük ve haftalık trend typeların eşleşmeşi 

In [1145]:
data_copy["mtf_bullish"] = (data_copy["trend_short"] == 1) & (data_copy["weekly_trend"] == 1)
data_copy["mtf_bearish"] = (data_copy["trend_short"] == -1) & (data_copy["weekly_trend"] == -1)

# Labeling gaps

## Common Gaps

In [1146]:
data_copy["common_gap"] = (
    data_copy["is_gap"] &
    (data_copy["normalized_gap_size"] < 0.5) &
    (~data_copy["high_volume"]) &
    (data_copy["trend_short"] == 0)
)

## Breakaway Gaps

In [1147]:
data_copy["breakaway_gap"] = (
    data_copy["is_gap"] &
    (data_copy["normalized_gap_size"] >= 0.8) &
    data_copy["high_volume"] &
    data_copy["consolidation"] &
    (
        (data_copy["gap_up"] & data_copy["breaks_resistance"] & data_copy["mtf_bullish"]) |
        (data_copy["gap_down"] & data_copy["breaks_support"] & data_copy["mtf_bearish"])
    )
)

## Runaway Gaps

In [1148]:
data_copy["runaway_gap"] = (
    data_copy["is_gap"] &
    (data_copy["normalized_gap_size"] >= 0.5) &
    data_copy["high_volume"] &
    (~data_copy["consolidation"]) &
    (
        (data_copy["gap_up"] & data_copy["mtf_bullish"] & data_copy["bullish_momentum"] & (~data_copy["extended_up_move"])) |
        (data_copy["gap_down"] & data_copy["mtf_bearish"] & data_copy["bearish_momentum"] & (~data_copy["extended_down_move"]))
    )
)

## Exhaustion Gaps

In [1149]:
data_copy["exhaustion_gap"] = (
    data_copy["is_gap"] &
    (data_copy["normalized_gap_size"] >= 0.8) &
    data_copy["very_high_volume"] &
    (
        (data_copy["gap_up"] & data_copy["extended_up_move"] & (data_copy["RSI"] > 70) & data_copy["near_resistance"]) |
        (data_copy["gap_down"] & data_copy["extended_down_move"] & (data_copy["RSI"] < 30) & data_copy["near_support"])
    )
)

## Tek sütunda birleştirme

In [1150]:
data_copy["gap_type"] = np.nan

data_copy.loc[data_copy["common_gap"], "gap_type"] = "Common Gap"
data_copy.loc[data_copy["breakaway_gap"], "gap_type"] = "Breakaway Gap"
data_copy.loc[data_copy["runaway_gap"], "gap_type"] = "Runaway Gap"
data_copy.loc[data_copy["exhaustion_gap"], "gap_type"] = "Exhaustion Gap"

/tmp/ipykernel_776/4120197614.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Common Gap' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  data_copy.loc[data_copy["common_gap"], "gap_type"] = "Common Gap"


## Öncelik Verme

In [1151]:
priority = ["Exhaustion Gap", "Breakaway Gap", "Runaway Gap", "Common Gap"]

for idx in data_copy.index[data_copy["is_gap"]]:
    labels = []
    if data_copy.loc[idx, "exhaustion_gap"]:
        labels.append("Exhaustion Gap")
    if data_copy.loc[idx, "breakaway_gap"]:
        labels.append("Breakaway Gap")
    if data_copy.loc[idx, "runaway_gap"]:
        labels.append("Runaway Gap")
    if data_copy.loc[idx, "common_gap"]:
        labels.append("Common Gap")

    if labels:
        for p in priority:
            if p in labels:
                data_copy.loc[idx, "gap_type"] = p
                break

# Görselleştirme

## Son support ve resistance

In [1152]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# GÖRSELLEŞTİRME (PLOTLY)
# ==========================================

print("\nGörselleştirme hazırlanıyor...")

# 1. Grafik Verisini Hazırla (Son dönem daha okunaklı olur)
plot_data_copy = data_copy.copy()  # Son 250 günü (yaklaşık 1 yıl) çizelim
# Eğer veri çoksa grafiği kasmaması için son kısmı seçmek iyidir.

# 2. Subplot oluştur (2 satır, 1 sütun: Üstte mumlar, altta hacim)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                    vertical_spacing=0.03, subplot_titles=(f'{ticker} Technical Analysis', 'Volume'), 
                    row_width=[0.2, 0.7]) # Alt grafiği küçült


# --- A. ÜST GRAFİK (CANDLESTICK & GÖSTERGELER) ---

# Candlestick (Mum Grafiği)
fig.add_trace(go.Candlestick(x=plot_data_copy.index,
                open=plot_data_copy['Open'],
                high=plot_data_copy['High'],
                low=plot_data_copy['Low'],
                close=plot_data_copy['Close'],
                name='Price'), row=1, col=1)

# Hareketli Ortalamalar
fig.add_trace(go.Scatter(x=plot_data_copy.index, y=plot_data_copy['MA20'], name='MA20', line=dict(color='blue', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=plot_data_copy.index, y=plot_data_copy['MA50'], name='MA50', line=dict(color='orange', width=1)), row=1, col=1)

# Support / Resistance Seviyeleri (Son günün değerlerini çizelim)
last_sup = plot_data_copy['rolling_support'].iloc[-1]
last_res = plot_data_copy['rolling_resistance'].iloc[-1]

fig.add_shape(type="line", x0=plot_data_copy.index[0], y0=last_sup, x1=plot_data_copy.index[-1], y1=last_sup,
            line=dict(color="green", width=2, dash="dash"), row=1, col=1)
fig.add_shape(type="line", x0=plot_data_copy.index[0], y0=last_res, x1=plot_data_copy.index[-1], y1=last_res,
            line=dict(color="red", width=2, dash="dash"), row=1, col=1)

# Destek/Direnç etiketlerini ekle
fig.add_annotation(x=plot_data_copy.index[-1], y=last_sup, text="Support", showarrow=False, yshift=10, row=1, col=1)
fig.add_annotation(x=plot_data_copy.index[-1], y=last_res, text="Resistance", showarrow=False, yshift=10, row=1, col=1)


# --- B. ARKA PLAN TREND RENKLENDİRMESİ ---

# Arka planı trend durumuna göre renklendirme (vrect)
current_trend = None
start_date = plot_data_copy.index[0]

for i in range(len(plot_data_copy)):
    date = plot_data_copy.index[i]
    trend = plot_data_copy['trend_short'].iloc[i]
    
    # Trend değiştiyse veya son barsak, bir önceki bölgeyi çiz
    if trend != current_trend or i == len(plot_data_copy) - 1:
        color = None
        if current_trend == 1: # Uptrend
            color = "rgba(0, 250, 0, 0.1)" # Çok açık yeşil
        elif current_trend == -1: # Downtrend
            color = "rgba(250, 0, 0, 0.1)" # Çok açık kırmızı
        
        # Renk varsa vrect ekle
        if color:
            fig.add_vrect(x0=start_date, x1=date, fillcolor=color, layer="below", line_width=0, row=1, col=1)
        
        # Yeni bölgeyi başlat
        start_date = date
        current_trend = trend


# --- C. GAPLERİ VURGULAMA VE ETİKETLEME ---

# Gap olan günleri filtrele
gaps_in_plot = plot_data_copy[plot_data_copy['is_gap'] & plot_data_copy['gap_type'].notna()]

# Gap tiplerine göre renkler
gap_colors = {
    "Common Gap": "gray",
    "Breakaway Gap": "blue",
    "Runaway Gap": "purple",
    "Exhaustion Gap": "orange"
}

for idx, gap in gaps_in_plot.iterrows():
    # Gap'in gerçekleştiği anı gösteren dikey çizgi
    gap_type = gap['gap_type']
    color = gap_colors.get(gap_type, "black") # Bilinmeyen tip olursa siyah yap
    
    fig.add_vline(x=idx, line_width=1.5, line_dash="dot", line_color=color, row=1, col=1)
    
    # Gap tipini yazan metin etiketi
    # Metni, gap yönüne göre mumun üstüne veya altına koy
    y_pos = gap['High'] * 1.02 if gap['gap_up'] else gap['Low'] * 0.98
    text_angle = -90 # Yazıyı dik yap
    
    fig.add_annotation(x=idx, y=y_pos, text=gap_type, showarrow=True, 
                        arrowhead=1, ax=0, ay=-40 if gap['gap_up'] else 40, # Ok yönü
                        font=dict(color=color, size=10), textangle=text_angle, row=1, col=1)


# --- D. ALT GRAFİK (HACİM) ---

# Hacim barları
fig.add_trace(go.Bar(x=plot_data_copy.index, y=plot_data_copy['Volume'], name='Volume', marker_color='gray'), row=2, col=1)
# Hacim MA
fig.add_trace(go.Scatter(x=plot_data_copy.index, y=plot_data_copy['VOL_MA20'], name='Volume MA20', line=dict(color='black', width=1)), row=2, col=1)


# --- E. GRAFİK AYARLARI ---

fig.update_layout(
    title=f'{ticker} Stocks Analysis (Gaps & Trends)',
    xaxis_title='Date',
    yaxis_title='Price',
    xaxis_rangeslider_visible=False, # Alttaki slider'ı kapat (subplot'ta çakışıyor)
    height=800, # Grafik yüksekliği
    template='plotly_white' # Temiz beyaz tema
)

# Y eksenlerini ayarla (üst grafik fiyat, alt grafik hacim)
fig.update_yaxes(title_text="Price ($)", row=1, col=1)
fig.update_yaxes(title_text="Volume", row=2, col=1)

# Grafiği göster
fig.show()

# İsteğe bağlı: Grafiği HTML dosyası olarak kaydet
# fig.write_html(f"{ticker}_gap_analizi.html")
# print(f"Grafik {ticker}_gap_analizi.html olarak kaydedildi.")


Görselleştirme hazırlanıyor...


## Tüm support ve resistance linelar

In [1153]:
print("\nGörselleştirme hazırlanıyor (Tüm Destek/Direnç Seviyeleri Dahil)...")

# 1. Grafik Verisini Hazırla
plot_data_copy = data_copy.copy().tail(500)

# 2. Subplot oluştur
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                    vertical_spacing=0.03, 
                    subplot_titles=(f'{ticker} Teknik Analiz', 'Hacim'), 
                    row_heights=[0.7, 0.3]) 

# --- A. ÜST GRAFİK ---

# Candlestick
fig.add_trace(go.Candlestick(
    x=plot_data_copy.index,
    open=plot_data_copy['Open'],
    high=plot_data_copy['High'],
    low=plot_data_copy['Low'],
    close=plot_data_copy['Close'],
    name='Fiyat'), row=1, col=1)

# Hareketli Ortalamalar
fig.add_trace(go.Scatter(x=plot_data_copy.index, y=plot_data_copy['MA20'], name='MA20', line=dict(color='rgba(0,0,255,0.7)', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=plot_data_copy.index, y=plot_data_copy['MA50'], name='MA50', line=dict(color='rgba(255,165,0,0.7)', width=1)), row=1, col=1)

# --- YENİ: TÜM DESTEK VE DİRENÇ SEVİYELERİ (Sürekli Çizgi) ---
fig.add_trace(go.Scatter(
    x=plot_data_copy.index, 
    y=plot_data_copy['rolling_support'], 
    name='Dinamik Destek', 
    line=dict(color='green', width=1.5, dash='dot'),
    mode='lines'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=plot_data_copy.index, 
    y=plot_data_copy['rolling_resistance'], 
    name='Dinamik Direnç', 
    line=dict(color='red', width=1.5, dash='dot'),
    mode='lines'
), row=1, col=1)

# --- B. ARKA PLAN TREND RENKLENDİRMESİ ---
# Trendin değiştiği yerleri (True/False) bulup sadece o tarihleri alıyoruz
df_trend = plot_data_copy['trend_short']
change_points = df_trend.index[df_trend.ne(df_trend.shift())].tolist()

# Listenin sonuna verinin bitiş tarihini ekleyelim ki son bölge de boyansın
if plot_data_copy.index[-1] not in change_points:
    change_points.append(plot_data_copy.index[-1])

start_date = plot_data_copy.index[0]
for end_date in change_points:
    # Başlangıç tarihindeki trend değerine bakıyoruz
    trend_val = plot_data_copy.loc[start_date, 'trend_short']
    
    color = None
    if trend_val == 1: 
        color = "rgba(0, 250, 0, 0.1)" # Yeşil
    elif trend_val == -1: 
        color = "rgba(250, 0, 0, 0.1)" # Kırmızı
    
    if color:
        fig.add_vrect(x0=start_date, x1=end_date, fillcolor=color, 
                      layer="below", line_width=0, row=1, col=1)
    
    # Bir sonraki bölgenin başlangıcı, bu bölgenin bitişidir
    start_date = end_date

# --- C. GAPLERİ VURGULAMA VE ETİKETLEME ---

# Gap olan günleri filtrele
gaps_in_plot = plot_data_copy[plot_data_copy['is_gap'] & plot_data_copy['gap_type'].notna()]

# Gap tiplerine göre renkler
gap_colors = {
    "Common Gap": "gray",
    "Breakaway Gap": "blue",
    "Runaway Gap": "purple",
    "Exhaustion Gap": "orange"
}

for idx, gap in gaps_in_plot.iterrows():
    # Gap'in gerçekleştiği anı gösteren dikey çizgi
    gap_type = gap['gap_type']
    color = gap_colors.get(gap_type, "black") # Bilinmeyen tip olursa siyah yap
    
    fig.add_vline(x=idx, line_width=1.5, line_dash="dot", line_color=color, row=1, col=1)
    
    # Gap tipini yazan metin etiketi
    # Metni, gap yönüne göre mumun üstüne veya altına koy
    y_pos = gap['High'] * 1.02 if gap['gap_up'] else gap['Low'] * 0.98
    text_angle = -90 # Yazıyı dik yap
    
    fig.add_annotation(x=idx, y=y_pos, text=gap_type, showarrow=True, 
                        arrowhead=1, ax=0, ay=-40 if gap['gap_up'] else 40, # Ok yönü
                        font=dict(color=color, size=10), textangle=text_angle, row=1, col=1)

# --- D. ALT GRAFİK (HACİM) ---
fig.add_trace(go.Bar(x=plot_data_copy.index, y=plot_data_copy['Volume'], name='Hacim', marker_color='rgba(128,128,128,0.5)'), row=2, col=1)
fig.add_trace(go.Scatter(x=plot_data_copy.index, y=plot_data_copy['VOL_MA20'], name='Hacim MA20', line=dict(color='black', width=1.5)), row=2, col=1)

# --- E. AYARLAR ---
fig.update_layout(
    title=f'{ticker} Kapsamlı Teknik Analiz',
    xaxis_rangeslider_visible=False,
    height=900,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()


Görselleştirme hazırlanıyor (Tüm Destek/Direnç Seviyeleri Dahil)...


In [1154]:
import yfinance as yf
import pandas as pd
import numpy as np
import talib.abstract as ta

def analyze_stock(ticker, start_date, end_date):
    # 1. Verileri Çekme
    print(f"{ticker} verileri yfinance üzerinden çekiliyor...")
    df = yf.download(ticker, start=start_date, end=end_date)
    
    # ------------------------------------------------------------------
    # KRİTİK DÜZELTME: TA-Lib Abstract API'si küçük harf sütun isimleri bekler.
    # yfinance'tan gelen sütun isimlerini (Open, High, vs.) küçük harfe (open, high, vs.) çeviriyoruz.
    # ------------------------------------------------------------------
    # Eğer yfinance MultiIndex dönüyorsa (örneğin sadece 'AAPL' çeksek bile bazen yapar),
    # sütun seviyelerini düzeltmek için:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.columns = df.columns.str.lower()
    
    # 2. Teknik Göstergeler (SMA ve RSI)
    df['SMA_20'] = df['close'].rolling(window=20).mean()
    df['SMA_50'] = df['close'].rolling(window=50).mean()
    
    # TA-Lib abstract kullanımı artık çok daha temiz. 
    # İçinde 'close' geçen tüm df'yi verdiğimizde o aradığını otomatik bulur.
    df['RSI'] = ta.RSI(df, timeperiod=14)
    
    df['Vol_SMA_20'] = df['volume'].rolling(window=20).mean()

    # 3. Trend Analizi (Up, Down, Sideway)
    df['Trend'] = 'Sideway'
    trend_threshold = 0.02 
    sma_diff = (df['SMA_20'] - df['SMA_50']) / df['SMA_50']
    
    df.loc[sma_diff > trend_threshold, 'Trend'] = 'Up'
    df.loc[sma_diff < -trend_threshold, 'Trend'] = 'Down'

    # 4. Geri Çekilme (Pullback) Noktaları (Sütunlar küçük harfe göre güncellendi)
    df['Pullback'] = False
    df.loc[(df['Trend'] == 'Up') & (df['low'] < df['SMA_20']) & (df['close'] > df['SMA_50']), 'Pullback'] = True
    df.loc[(df['Trend'] == 'Down') & (df['high'] > df['SMA_20']) & (df['close'] < df['SMA_50']), 'Pullback'] = True

    # 5. Pivot Noktaları, Destek (Support) ve Direnç (Resistance)
    df['Pivot'] = (df['high'].shift(1) + df['low'].shift(1) + df['close'].shift(1)) / 3
    df['R1'] = (2 * df['Pivot']) - df['low'].shift(1)
    df['S1'] = (2 * df['Pivot']) - df['high'].shift(1)
    
    df['Sideway_Support'] = np.where(df['Trend'] == 'Sideway', df['S1'], np.nan)
    df['Sideway_Resistance'] = np.where(df['Trend'] == 'Sideway', df['R1'], np.nan)

    # 6. Gap (Boşluk) Analizi ve Sınıflandırma
    # Pandas FutureWarning hatasını çözmek için 'object' (string) tipinde başlatıyoruz.
    df['Gap_Type'] = pd.Series('None', index=df.index, dtype='object')
    
    gap_up = df['low'] > df['high'].shift(1)
    gap_down = df['high'] < df['low'].shift(1)
    is_gap = gap_up | gap_down

    prev_trend = df['Trend'].shift(1)
    
    for i in range(1, len(df)):
        if is_gap.iloc[i]:
            current_trend_val = df['Trend'].iloc[i]
            prev_trend_val = prev_trend.iloc[i]
            rsi_val = df['RSI'].iloc[i]
            vol_spike = df['volume'].iloc[i] > (1.5 * df['Vol_SMA_20'].iloc[i]) 
            
            if prev_trend_val == 'Sideway' and current_trend_val == 'Sideway':
                df.iloc[i, df.columns.get_loc('Gap_Type')] = 'Common Gap'
            elif prev_trend_val == 'Sideway' and current_trend_val != 'Sideway' and vol_spike:
                df.iloc[i, df.columns.get_loc('Gap_Type')] = 'Breakaway Gap'
            elif (prev_trend_val == 'Up' and rsi_val > 70 and vol_spike) or \
                 (prev_trend_val == 'Down' and rsi_val < 30 and vol_spike):
                df.iloc[i, df.columns.get_loc('Gap_Type')] = 'Exhaustion Gap'
            elif (prev_trend_val == 'Up' and current_trend_val == 'Up') or \
                 (prev_trend_val == 'Down' and current_trend_val == 'Down'):
                df.iloc[i, df.columns.get_loc('Gap_Type')] = 'Runaway Gap'
            else:
                df.iloc[i, df.columns.get_loc('Gap_Type')] = 'Unclassified Gap'

    return df

# Örnek Kullanım
if __name__ == "__main__":
    pd.set_option('display.max_columns', None)
    
    # Hisseden veriyi çek ve analiz et
    result_df = analyze_stock("AAPL", "2023-01-01", "2024-01-01")
    
    # Sadece aksiyon olan günleri (Pullback veya Gap) filtreleyip ekrana yazdıralım
    action_days = result_df[(result_df['Pullback'] == True) | (result_df['Gap_Type'] != 'None')].copy()
    
    print("\n--- Önemli Hareketlerin Olduğu Günler (Pullback & Gaps) ---")
    columns_to_show = ['close', 'Trend', 'Pullback', 'Sideway_Support', 'Sideway_Resistance', 'Gap_Type']
    print(action_days[columns_to_show].tail(20))

[*********************100%***********************]  1 of 1 completed

AAPL verileri yfinance üzerinden çekiliyor...

--- Önemli Hareketlerin Olduğu Günler (Pullback & Gaps) ---
Price            close    Trend  Pullback  Sideway_Support  \
Date                                                         
2023-08-28  178.102722     Down      True              NaN   
2023-08-29  181.987167     Down      True              NaN   
2023-09-06  180.791183     Down      True              NaN   
2023-09-07  175.503189     Down     False              NaN   
2023-10-05  172.883881     Down      True              NaN   
2023-10-06  175.434006     Down      True              NaN   
2023-10-09  176.916580     Down      True              NaN   
2023-10-10  176.323532     Down      True              NaN   
2023-11-02  175.513077  Sideway     False       169.332158   
2023-11-14  185.512817  Sideway     False       182.104893   
2023-12-04  187.482330       Up      True              NaN   
2023-12-05  191.431305       Up     False              NaN   
2023-12-11  191.193771   

In [1155]:
import yfinance as yf
import pandas as pd
import numpy as np
import talib.abstract as ta
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def analyze_stock(ticker, start_date, end_date):
    print(f"{ticker} verileri yfinance üzerinden çekiliyor...")
    df = yf.download(ticker, start=start_date, end=end_date)
    
    # Sütun isimlerini küçük harfe çevirme (TA-Lib uyumluluğu için)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.columns = df.columns.str.lower()
    
    # 2. Teknik Göstergeler
    df['SMA_20'] = df['close'].rolling(window=20).mean()
    df['SMA_50'] = df['close'].rolling(window=50).mean()
    df['RSI'] = ta.RSI(df, timeperiod=14)
    df['Vol_SMA_20'] = df['volume'].rolling(window=20).mean()

    # 3. Trend Analizi
    df['Trend'] = 'Sideway'
    trend_threshold = 0.02 
    sma_diff = (df['SMA_20'] - df['SMA_50']) / df['SMA_50']
    df.loc[sma_diff > trend_threshold, 'Trend'] = 'Up'
    df.loc[sma_diff < -trend_threshold, 'Trend'] = 'Down'

    # 4. Pullback (Geri Çekilme)
    df['Pullback'] = False
    df.loc[(df['Trend'] == 'Up') & (df['low'] < df['SMA_20']) & (df['close'] > df['SMA_50']), 'Pullback'] = True
    df.loc[(df['Trend'] == 'Down') & (df['high'] > df['SMA_20']) & (df['close'] < df['SMA_50']), 'Pullback'] = True

    # 5. Pivot Noktaları (Sadece Sideway)
    df['Pivot'] = (df['high'].shift(1) + df['low'].shift(1) + df['close'].shift(1)) / 3
    df['R1'] = (2 * df['Pivot']) - df['low'].shift(1)
    df['S1'] = (2 * df['Pivot']) - df['high'].shift(1)
    
    df['Sideway_Support'] = np.where(df['Trend'] == 'Sideway', df['S1'], np.nan)
    df['Sideway_Resistance'] = np.where(df['Trend'] == 'Sideway', df['R1'], np.nan)

    # 6. Gap Analizi
    df['Gap_Type'] = pd.Series('None', index=df.index, dtype='object')
    is_gap = (df['low'] > df['high'].shift(1)) | (df['high'] < df['low'].shift(1))
    prev_trend = df['Trend'].shift(1)
    
    for i in range(1, len(df)):
        if is_gap.iloc[i]:
            current_trend = df['Trend'].iloc[i]
            prev_tr = prev_trend.iloc[i]
            rsi = df['RSI'].iloc[i]
            vol_spike = df['volume'].iloc[i] > (1.5 * df['Vol_SMA_20'].iloc[i]) 
            
            if prev_tr == 'Sideway' and current_trend == 'Sideway':
                df.iloc[i, df.columns.get_loc('Gap_Type')] = 'Common Gap'
            elif prev_tr == 'Sideway' and current_trend != 'Sideway' and vol_spike:
                df.iloc[i, df.columns.get_loc('Gap_Type')] = 'Breakaway Gap'
            elif (prev_tr == 'Up' and rsi > 70 and vol_spike) or (prev_tr == 'Down' and rsi < 30 and vol_spike):
                df.iloc[i, df.columns.get_loc('Gap_Type')] = 'Exhaustion Gap'
            elif (prev_tr == 'Up' and current_trend == 'Up') or (prev_tr == 'Down' and current_trend == 'Down'):
                df.iloc[i, df.columns.get_loc('Gap_Type')] = 'Runaway Gap'
            else:
                df.iloc[i, df.columns.get_loc('Gap_Type')] = 'Unclassified Gap'

    return df

def plot_interactive_chart(df, ticker):
    print("Grafik hazırlanıyor...")
    df= df.copy().tail(500)
    
    # 2 satırlı bir figür oluştur (Üstte Fiyat, Altta Hacim)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                        vertical_spacing=0.05, 
                        row_heights=[0.75, 0.25],
                        subplot_titles=(f'{ticker} Kapsamlı Teknik Analiz', 'Hacim'))

    # 1. Mum Grafiği (Candlestick)
    fig.add_trace(go.Candlestick(x=df.index, open=df['open'], high=df['high'], 
                                 low=df['low'], close=df['close'], name='Fiyat'), row=1, col=1)

    # 2. Hareketli Ortalamalar
    fig.add_trace(go.Scatter(x=df.index, y=df['SMA_20'], line=dict(color='blue', width=1.5), name='SMA 20'), row=1, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['SMA_50'], line=dict(color='orange', width=1.5), name='SMA 50'), row=1, col=1)

    # 3. Sideway Destek ve Dirençleri (Kesik Çizgiler)
    fig.add_trace(go.Scatter(x=df.index, y=df['Sideway_Support'], mode='lines', 
                             line=dict(color='green', width=1, dash='dot'), name='Dinamik Destek (S1)'), row=1, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['Sideway_Resistance'], mode='lines', 
                             line=dict(color='red', width=1, dash='dot'), name='Dinamik Direnç (R1)'), row=1, col=1)

    # 4. Pullback (Geri Çekilme) İşaretçileri
    pullbacks = df[df['Pullback']]
    fig.add_trace(go.Scatter(x=pullbacks.index, y=pullbacks['low'] * 0.98, mode='markers',
                             marker=dict(symbol='triangle-up', size=12, color='purple', line=dict(width=1, color='black')),
                             name='Pullback'), row=1, col=1)

    # 5. Hacim (Volume) Bar Grafiği ve Hacim Ortalaması
    colors = ['green' if row['close'] >= row['open'] else 'red' for idx, row in df.iterrows()]
    fig.add_trace(go.Bar(x=df.index, y=df['volume'], marker_color=colors, name='Hacim', opacity=0.7), row=2, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['Vol_SMA_20'], line=dict(color='black', width=1.5), name='Hacim SMA 20'), row=2, col=1)

    # 6. Trendlere Göre Arka Plan Renklendirmesi (Up=Yeşil, Down=Kırmızı, Sideway=Gri)
    trend_blocks = []
    current_trend = df['Trend'].iloc[0]
    start_date = df.index[0]
    
    for date, trend in df['Trend'].items():
        if trend != current_trend:
            trend_blocks.append((start_date, date, current_trend))
            current_trend = trend
            start_date = date
    trend_blocks.append((start_date, df.index[-1], current_trend))

    color_map = {'Up': 'rgba(0, 255, 0, 0.05)', 'Down': 'rgba(255, 0, 0, 0.05)', 'Sideway': 'rgba(128, 128, 128, 0.05)'}
    for start, end, trend in trend_blocks:
        fig.add_vrect(x0=start, x1=end, fillcolor=color_map.get(trend, 'white'), opacity=1, layer="below", line_width=0, row=1, col=1)

    # 7. Gap (Boşluk) Etiketleri Ekleme
    gaps = df[df['Gap_Type'] != 'None']
    for idx, row in gaps.iterrows():
        # Gap türüne göre farklı renkler atayabiliriz
        bg_color = "black"
        if "Breakaway" in row['Gap_Type']: bg_color = "blue"
        elif "Exhaustion" in row['Gap_Type']: bg_color = "red"
        elif "Runaway" in row['Gap_Type']: bg_color = "green"

        fig.add_annotation(x=idx, y=row['high'], text=row['Gap_Type'],
                           showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor=bg_color,
                           font=dict(color="white", size=10), bgcolor=bg_color, opacity=0.8, yshift=10, row=1, col=1)

    # Grafik Ayarları
    fig.update_layout(
        xaxis_rangeslider_visible=False, # Grafikteki ekstra slider'ı kapatır
        template="plotly_dark", # İstersen "plotly_white" yapabilirsin
        hovermode="x unified",
        height=800,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    
    # Boş haftasonu (işlem görmeyen günleri) eksenden kaldır
    fig.update_xaxes(rangebreaks=[dict(bounds=["sat", "mon"])]) 

    # Tarayıcıda aç
    fig.show()

# Örnek Kullanım (Kendi hisseni ve tarihlerini buraya yazabilirsin)
if __name__ == "__main__":
    ticker_symbol = "NVDA"  # Veya "ISCTR.IS", "AAPL" vs.
    df_analyzed = analyze_stock(ticker_symbol, "2015-01-01", "2026-04-05")
    plot_interactive_chart(df_analyzed, ticker_symbol)

[*********************100%***********************]  1 of 1 completed

NVDA verileri yfinance üzerinden çekiliyor...
Grafik hazırlanıyor...


In [1157]:
import yfinance as yf
import pandas as pd
import numpy as np
import talib.abstract as ta
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# atr_multiplier parametresini ekledik. Default olarak 0.5 (ATR'nin yarısı) kullanıyoruz.
def analyze_stock(ticker, start_date, end_date, atr_multiplier=0.5):
    print(f"{ticker} verileri yfinance üzerinden çekiliyor...")
    df = yf.download(ticker, start=start_date, end=end_date)
    
    # Sütun isimlerini TA-Lib uyumluluğu için küçük harfe çevirme
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.columns = df.columns.str.lower()
    
    # 2. Teknik Göstergeler ve ATR
    df['SMA_20'] = df['close'].rolling(window=20).mean()
    df['SMA_50'] = df['close'].rolling(window=50).mean()
    df['RSI'] = ta.RSI(df, timeperiod=14)
    df['Vol_SMA_20'] = df['volume'].rolling(window=20).mean()
    
    # ATR (Average True Range) Hesaplaması
    df['ATR'] = ta.ATR(df, timeperiod=14)

    # 3. DİNAMİK TREND ANALİZİ (Statik 0.02 yerine ATR kullanıyoruz)
    df['Trend'] = 'Sideway'
    
    # ATR'yi fiyata bölerek yüzdesel hale getiriyoruz ve çarpanımızla (0.5) çarpıyoruz
    df['Dynamic_Threshold'] = (df['ATR'] / df['close']) * atr_multiplier
    
    sma_diff = (df['SMA_20'] - df['SMA_50']) / df['SMA_50']
    
    # SMA farkı, o günkü dinamik ATR eşiğinden büyükse trend değişir
    df.loc[sma_diff > df['Dynamic_Threshold'], 'Trend'] = 'Up'
    df.loc[sma_diff < -df['Dynamic_Threshold'], 'Trend'] = 'Down'

    # 4. Pullback (Geri Çekilme)
    df['Pullback'] = False
    df.loc[(df['Trend'] == 'Up') & (df['low'] < df['SMA_20']) & (df['close'] > df['SMA_50']), 'Pullback'] = True
    df.loc[(df['Trend'] == 'Down') & (df['high'] > df['SMA_20']) & (df['close'] < df['SMA_50']), 'Pullback'] = True

    # 5. Pivot Noktaları (Sadece Sideway)
    df['Pivot'] = (df['high'].shift(1) + df['low'].shift(1) + df['close'].shift(1)) / 3
    df['R1'] = (2 * df['Pivot']) - df['low'].shift(1)
    df['S1'] = (2 * df['Pivot']) - df['high'].shift(1)
    
    df['Sideway_Support'] = np.where(df['Trend'] == 'Sideway', df['S1'], np.nan)
    df['Sideway_Resistance'] = np.where(df['Trend'] == 'Sideway', df['R1'], np.nan)

    # 6. Gap Analizi
    df['Gap_Type'] = pd.Series('None', index=df.index, dtype='object')
    is_gap = (df['low'] > df['high'].shift(1)) | (df['high'] < df['low'].shift(1))
    prev_trend = df['Trend'].shift(1)
    
    for i in range(1, len(df)):
        if is_gap.iloc[i]:
            current_trend = df['Trend'].iloc[i]
            prev_tr = prev_trend.iloc[i]
            rsi = df['RSI'].iloc[i]
            vol_spike = df['volume'].iloc[i] > (1.5 * df['Vol_SMA_20'].iloc[i]) 
            
            if prev_tr == 'Sideway' and current_trend == 'Sideway':
                df.iloc[i, df.columns.get_loc('Gap_Type')] = 'Common Gap'
            elif prev_tr == 'Sideway' and current_trend != 'Sideway' and vol_spike:
                df.iloc[i, df.columns.get_loc('Gap_Type')] = 'Breakaway Gap'
            elif (prev_tr == 'Up' and rsi > 70 and vol_spike) or (prev_tr == 'Down' and rsi < 30 and vol_spike):
                df.iloc[i, df.columns.get_loc('Gap_Type')] = 'Exhaustion Gap'
            elif (prev_tr == 'Up' and current_trend == 'Up') or (prev_tr == 'Down' and current_trend == 'Down'):
                df.iloc[i, df.columns.get_loc('Gap_Type')] = 'Runaway Gap'
            else:
                df.iloc[i, df.columns.get_loc('Gap_Type')] = 'Unclassified Gap'

    return df

def plot_interactive_chart(df, ticker):
    print("Grafik hazırlanıyor...")
    df= df.copy().tail(500)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                        vertical_spacing=0.05, 
                        row_heights=[0.75, 0.25],
                        subplot_titles=(f'{ticker} Kapsamlı Teknik Analiz (Dinamik ATR Eşikli)', 'Hacim'))

    # 1. Mum Grafiği
    fig.add_trace(go.Candlestick(x=df.index, open=df['open'], high=df['high'], 
                                 low=df['low'], close=df['close'], name='Fiyat'), row=1, col=1)

    # 2. Hareketli Ortalamalar
    fig.add_trace(go.Scatter(x=df.index, y=df['SMA_20'], line=dict(color='blue', width=1.5), name='SMA 20'), row=1, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['SMA_50'], line=dict(color='orange', width=1.5), name='SMA 50'), row=1, col=1)

    # 3. Sideway Destek ve Dirençleri
    fig.add_trace(go.Scatter(x=df.index, y=df['Sideway_Support'], mode='lines', 
                             line=dict(color='green', width=1, dash='dot'), name='Dinamik Destek (S1)'), row=1, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['Sideway_Resistance'], mode='lines', 
                             line=dict(color='red', width=1, dash='dot'), name='Dinamik Direnç (R1)'), row=1, col=1)

    # 4. Pullback İşaretçileri
    pullbacks = df[df['Pullback']]
    fig.add_trace(go.Scatter(x=pullbacks.index, y=pullbacks['low'] * 0.98, mode='markers',
                             marker=dict(symbol='triangle-up', size=12, color='purple', line=dict(width=1, color='black')),
                             name='Pullback'), row=1, col=1)

    # 5. Hacim
    colors = ['green' if row['close'] >= row['open'] else 'red' for idx, row in df.iterrows()]
    fig.add_trace(go.Bar(x=df.index, y=df['volume'], marker_color=colors, name='Hacim', opacity=0.7), row=2, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['Vol_SMA_20'], line=dict(color='black', width=1.5), name='Hacim SMA 20'), row=2, col=1)

    # 6. Arka Plan Renklendirmesi (Dinamik Trende Göre)
    trend_blocks = []
    current_trend = df['Trend'].iloc[0]
    start_date = df.index[0]
    
    for date, trend in df['Trend'].items():
        if trend != current_trend:
            trend_blocks.append((start_date, date, current_trend))
            current_trend = trend
            start_date = date
    trend_blocks.append((start_date, df.index[-1], current_trend))

    color_map = {'Up': 'rgba(0, 255, 0, 0.05)', 'Down': 'rgba(255, 0, 0, 0.05)', 'Sideway': 'rgba(128, 128, 128, 0.05)'}
    for start, end, trend in trend_blocks:
        fig.add_vrect(x0=start, x1=end, fillcolor=color_map.get(trend, 'white'), opacity=1, layer="below", line_width=0, row=1, col=1)

    # 7. Gap Etiketleri
    gaps = df[df['Gap_Type'] != 'None']
    for idx, row in gaps.iterrows():
        bg_color = "black"
        if "Breakaway" in row['Gap_Type']: bg_color = "blue"
        elif "Exhaustion" in row['Gap_Type']: bg_color = "red"
        elif "Runaway" in row['Gap_Type']: bg_color = "green"

        fig.add_annotation(x=idx, y=row['high'], text=row['Gap_Type'],
                           showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor=bg_color,
                           font=dict(color="white", size=10), bgcolor=bg_color, opacity=0.8, yshift=10, row=1, col=1)

    fig.update_layout(
        xaxis_rangeslider_visible=False,
        template="plotly_dark",
        hovermode="x unified",
        height=800,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    
    fig.update_xaxes(rangebreaks=[dict(bounds=["sat", "mon"])]) 
    fig.show()

# Örnek Kullanım
if __name__ == "__main__":
    ticker_symbol = "AAPL"
    
    # atr_multiplier=0.5 diyerek eşiği ATR'nin %50'sine ayarladık. 
    # Çok sığ piyasalarda bunu 1.0 yapabilir, çok stabil piyasalarda 0.2'ye indirebilirsin.
    df_analyzed = analyze_stock(ticker_symbol, "2015-01-01", "2026-04-05", atr_multiplier=0.5)
    
    plot_interactive_chart(df_analyzed, ticker_symbol)

AAPL verileri yfinance üzerinden çekiliyor...


[*********************100%***********************]  1 of 1 completed


Grafik hazırlanıyor...
